# Atlas Ledger — Receipt NER Model Training

Trains a DistilBERT token classifier to extract fields from receipt OCR text.
Exports a quantized INT8 ONNX model for on-device inference via `onnxruntime-react-native`.

**Runtime:** ~3–4 hours on Colab free T4  
**Output:** `receipt-classifier.onnx`, `vocab.txt`, `label_map.json`

## BIO Tag Schema
| Tag | Field |
|-----|-------|
| B-MER / I-MER | Merchant name |
| B-TOT / I-TOT | Grand total amount |
| B-TAX / I-TAX | Tax amount |
| B-TIP / I-TIP | Tip amount |
| B-DATE | Date (single token) |
| O | Outside / other |

In [ ]:
# @title 1. Install dependencies
# Install optimum first — pip resolves a compatible transformers version automatically.
# Do NOT pin transformers separately; let optimum pick the right version.
!pip install -q "optimum[onnxruntime]>=1.17.0"
!pip install -q "datasets>=2.19.0" "seqeval==1.2.2" "accelerate>=0.29.0"

# Verify versions
import importlib
for pkg in ["optimum", "transformers", "datasets", "onnxruntime"]:
    try:
        print(f"{pkg}: {importlib.import_module(pkg).__version__}")
    except Exception:
        print(f"{pkg}: not yet imported")

In [ ]:
# @title 2. Imports and constants
import json
import os
import re
import shutil
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from seqeval.metrics import f1_score, classification_report

# ── Label schema ─────────────────────────────────────────────────────────────
# Must match ReceiptONNXService.ts BIO tag constants exactly.
LABEL_LIST = [
    "O",
    "B-MER", "I-MER",
    "B-TOT", "I-TOT",
    "B-TAX", "I-TAX",
    "B-TIP", "I-TIP",
    "B-DATE",
]
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)  # 10

MODEL_CHECKPOINT = "distilbert-base-uncased"
OUTPUT_DIR = Path("./receipt-ner-model")
ONNX_DIR = Path("./receipt-ner-onnx")
RELEASE_DIR = Path("./release")

print(f"Labels ({NUM_LABELS}): {LABEL_LIST}")
print(f"GPU available: {torch.cuda.is_available()}")

## 3. Data Loading

We use two open datasets:
- **CORD v2** — 11,000 receipts from Hugging Face (automatic download)
- **SROIE** — 1,000 ICDAR competition receipts from Hugging Face mirror (automatic download)

Both are converted to the same BIO token format.

In [ ]:
# @title 3a. CORD v2 — load and inspect
cord_raw = load_dataset("naver-clova-ix/cord-v2")
print(cord_raw)
# Inspect one example to understand structure
ex = cord_raw["train"][0]
print("\nCORD example keys:", list(ex.keys()))
ground_truth = json.loads(ex["ground_truth"])
print("ground_truth keys:", list(ground_truth.keys()))

In [ ]:
# @title 3b. SROIE — load from Hugging Face mirror
# darentang/sroie mirrors the ICDAR 2019 SROIE dataset with key-value annotations.
sroie_raw = load_dataset("darentang/sroie", trust_remote_code=True)
print(sroie_raw)
ex_s = sroie_raw["train"][0]
print("\nSROIE example keys:", list(ex_s.keys()))

In [ ]:
# @title 4. BIO conversion utilities

# ── Amount normalizer ─────────────────────────────────────────────────────────
_AMOUNT_RE = re.compile(r"[\$,\s]")  # strip $, commas, spaces from amounts

def normalize_amount(s: str) -> Optional[float]:
    """Parse a string like '$1,234.56' → 1234.56, or None if unparseable."""
    try:
        return float(_AMOUNT_RE.sub("", s))
    except (ValueError, TypeError):
        return None

# ── Token-level BIO tagger ────────────────────────────────────────────────────
def tag_tokens(tokens: list[str], spans: dict[str, str]) -> list[str]:
    """
    Given a whitespace-split token list and a dict of {bio_prefix: span_text},
    assign BIO labels by sliding-window substring matching.

    spans = {
        "MER": "STARBUCKS COFFEE",
        "TOT": "15.60",
        "TAX": "0.87",
        "TIP": None,
        "DATE": "12/05/2025",
    }
    """
    labels = ["O"] * len(tokens)

    for field, span_text in spans.items():
        if not span_text:
            continue
        span_tokens = span_text.upper().split()
        n = len(span_tokens)
        if n == 0:
            continue

        # Slide over tokens looking for a match
        for i in range(len(tokens) - n + 1):
            window = [t.upper() for t in tokens[i : i + n]]
            if window == span_tokens:
                labels[i] = f"B-{field}"
                for j in range(1, n):
                    labels[i + j] = f"I-{field}"
                break  # first match wins

    return labels


def tokens_and_labels_to_example(tokens: list[str], labels: list[str]) -> dict:
    """Convert parallel token/label lists to model-ready dict."""
    assert len(tokens) == len(labels), f"{len(tokens)} tokens vs {len(labels)} labels"
    return {
        "tokens": tokens,
        "ner_tags": [LABEL2ID[l] for l in labels],
    }

print("BIO utilities loaded.")

In [ ]:
# @title 5. Convert CORD v2 → BIO examples

def cord_to_bio(example: dict) -> Optional[dict]:
    """Convert a single CORD example to BIO token format."""
    try:
        gt = json.loads(example["ground_truth"])
        gt_parse = gt.get("gt_parse", {})

        # Extract OCR text lines from the image annotation
        # CORD provides words in the annotation field
        ann = example.get("meta", {}) or {}
        # Fallback: use the raw OCR text if available
        # CORD stores token text in image annotations; we reconstruct from gt fields
        # for training purposes we use the parsed ground-truth text directly.

        # Build a pseudo-OCR line from all text values in the parse tree
        all_text_parts = []

        def collect_text(obj):
            if isinstance(obj, str):
                all_text_parts.append(obj)
            elif isinstance(obj, list):
                for item in obj:
                    collect_text(item)
            elif isinstance(obj, dict):
                for v in obj.values():
                    collect_text(v)

        collect_text(gt_parse)
        full_text = " ".join(all_text_parts)
        tokens = full_text.split()
        if len(tokens) < 3:
            return None

        # Extract field spans
        total_obj = gt_parse.get("total", {})
        sub_total_obj = gt_parse.get("sub_total", {})
        menu_items = gt_parse.get("menu", [])
        meta_obj = gt_parse.get("meta", {})

        # Merchant: meta.shop.nm or first menu item company
        merchant = None
        if isinstance(meta_obj, dict):
            shop = meta_obj.get("shop", {})
            if isinstance(shop, dict):
                merchant = shop.get("nm")
        if not merchant and isinstance(meta_obj, dict):
            merchant = meta_obj.get("nm")

        # Total
        total_price = None
        if isinstance(total_obj, dict):
            total_price = total_obj.get("total_price") or total_obj.get("cashprice")

        # Tax
        tax_price = None
        if isinstance(sub_total_obj, dict):
            tax_price = sub_total_obj.get("tax_price")
        if not tax_price and isinstance(total_obj, dict):
            tax_price = total_obj.get("tax_price")

        # Tip
        tip_price = None
        if isinstance(sub_total_obj, dict):
            tip_price = sub_total_obj.get("etc_ttl") or sub_total_obj.get("tip")

        # Date
        date_str = None
        if isinstance(meta_obj, dict):
            date_str = meta_obj.get("date")

        spans = {
            "MER": str(merchant) if merchant else None,
            "TOT": str(total_price) if total_price else None,
            "TAX": str(tax_price) if tax_price else None,
            "TIP": str(tip_price) if tip_price else None,
            "DATE": str(date_str) if date_str else None,
        }

        labels = tag_tokens(tokens, spans)
        return tokens_and_labels_to_example(tokens, labels)

    except Exception:
        return None


cord_examples = []
for split in ["train", "validation", "test"]:
    if split not in cord_raw:
        continue
    for ex in cord_raw[split]:
        converted = cord_to_bio(ex)
        if converted:
            cord_examples.append(converted)

print(f"CORD: {len(cord_examples)} usable examples")

In [ ]:
# @title 6. Convert SROIE → BIO examples

def sroie_to_bio(example: dict) -> Optional[dict]:
    """Convert a single SROIE example to BIO token format."""
    try:
        # SROIE (darentang) stores: words (list), labels (list), bboxes, etc.
        # Labels are: 'company', 'date', 'address', 'total', 'other'
        words = example.get("words", []) or []
        word_labels = example.get("labels", []) or []

        if not words:
            # Fallback: some versions store as text/ner_tags
            words = example.get("tokens", []) or []
            word_labels = []

        if not words:
            return None

        # Map SROIE labels → our BIO tags
        # SROIE has word-level labels already; convert to BIO
        sroie_to_bio_map = {
            "company": "MER",
            "total": "TOT",
            "date": "DATE",
            "address": None,  # no corresponding tag — treat as O
            "other": None,
            "O": None,
        }

        if word_labels:
            # Convert word-level SROIE labels to BIO
            bio_labels = []
            prev_field = None
            for word, wlabel in zip(words, word_labels):
                field = sroie_to_bio_map.get(wlabel.lower(), None)
                if field is None:
                    bio_labels.append("O")
                    prev_field = None
                elif field == prev_field:
                    bio_labels.append(f"I-{field}")
                else:
                    bio_labels.append(f"B-{field}")
                    prev_field = field
        else:
            # No labels — all O (still useful as negative examples)
            bio_labels = ["O"] * len(words)

        if len(words) < 3:
            return None

        return tokens_and_labels_to_example(words, bio_labels)

    except Exception:
        return None


sroie_examples = []
for split in ["train", "test"]:
    if split not in sroie_raw:
        continue
    for ex in sroie_raw[split]:
        converted = sroie_to_bio(ex)
        if converted:
            sroie_examples.append(converted)

print(f"SROIE: {len(sroie_examples)} usable examples")

In [ ]:
# @title 7. Merge datasets and split
import random

random.seed(42)
all_examples = cord_examples + sroie_examples
random.shuffle(all_examples)

n = len(all_examples)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)
# remainder → test

train_data = all_examples[:n_train]
val_data   = all_examples[n_train : n_train + n_val]
test_data  = all_examples[n_train + n_val :]

print(f"Total: {n}  |  Train: {len(train_data)}  |  Val: {len(val_data)}  |  Test: {len(test_data)}")

# Convert to HuggingFace Dataset
def to_hf_dataset(examples: list[dict]) -> Dataset:
    return Dataset.from_dict({
        "tokens": [e["tokens"] for e in examples],
        "ner_tags": [e["ner_tags"] for e in examples],
    })

dataset = DatasetDict({
    "train": to_hf_dataset(train_data),
    "validation": to_hf_dataset(val_data),
    "test": to_hf_dataset(test_data),
})
print(dataset)

In [ ]:
# @title 8. Tokenize with subword alignment
#
# DistilBERT uses WordPiece which splits words into subword tokens.
# We propagate the BIO label of a word to its first subword token,
# and mark continuation subword tokens with -100 (ignored by loss).

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True,
    )

    aligned_labels = []
    for i, label_ids in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_row = []
        for word_id in word_ids:
            if word_id is None:
                # [CLS] and [SEP] tokens
                label_row.append(-100)
            elif word_id != prev_word_id:
                # First subword of a new word → use the word's label
                label_row.append(label_ids[word_id])
            else:
                # Continuation subword → ignore in loss
                label_row.append(-100)
            prev_word_id = word_id
        aligned_labels.append(label_row)

    tokenized["labels"] = aligned_labels
    return tokenized


tokenized_dataset = dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print(tokenized_dataset)

In [ ]:
# @title 9. Model, metrics, training config

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

data_collator = DataCollatorForTokenClassification(tokenizer)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [ID2LABEL[l] for l in label_row if l != -100]
        for label_row in labels
    ]
    true_preds = [
        [ID2LABEL[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]

    return {
        "f1": f1_score(true_labels, true_preds),
    }


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=12,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# @title 10. Train
# ~3-4 hours on Colab T4. Make sure Runtime → Change runtime type → GPU is set.
trainer.train()

In [ ]:
# @title 11. Evaluate on hold-out test set
test_results = trainer.predict(tokenized_dataset["test"])
predictions = np.argmax(test_results.predictions, axis=-1)
labels = test_results.label_ids

true_labels = [
    [ID2LABEL[l] for l in label_row if l != -100]
    for label_row in labels
]
true_preds = [
    [ID2LABEL[p] for p, l in zip(pred_row, label_row) if l != -100]
    for pred_row, label_row in zip(predictions, labels)
]

print(classification_report(true_labels, true_preds))

# ── Acceptance thresholds ─────────────────────────────────────────────────────
# These are minimum F1 scores for each field.
# The model should NOT be released if any threshold is not met.
THRESHOLDS = {
    "MER": 0.85,
    "TOT": 0.92,
    "TAX": 0.75,
    "TIP": 0.70,
    "DATE": 0.88,
}

from seqeval.metrics import classification_report as cr
report_dict = cr(true_labels, true_preds, output_dict=True)
passed = True
for field, threshold in THRESHOLDS.items():
    # seqeval uses full entity names
    entity_key = field  # seqeval strips B-/I- and reports by entity
    f1 = report_dict.get(entity_key, {}).get("f1-score", 0.0)
    status = "✅" if f1 >= threshold else "❌"
    if f1 < threshold:
        passed = False
    print(f"{status}  {field}: F1={f1:.3f}  (threshold {threshold})")

if not passed:
    print("\n⚠️  One or more fields below threshold. Consider more epochs or data augmentation.")
else:
    print("\n✅  All thresholds met — model ready for export.")

In [ ]:
# @title 12. Save best model checkpoint
OUTPUT_DIR.mkdir(exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Model saved to {OUTPUT_DIR}")
!ls -lh {OUTPUT_DIR}

In [ ]:
# @title 13. Export to ONNX and quantize (INT8)
#
# optimum handles the export and INT8 static quantization in one step.
# Output: receipt-ner-onnx/model_quantized.onnx  (~65 MB)

ONNX_DIR.mkdir(exist_ok=True)

# Step 1: Export to ONNX (FP32 first)
!optimum-cli export onnx \
    --model {OUTPUT_DIR} \
    --task token-classification \
    {ONNX_DIR}/fp32/

# Step 2: Quantize to INT8
!optimum-cli onnxruntime quantize \
    --onnx_model {ONNX_DIR}/fp32/ \
    --output {ONNX_DIR}/int8/ \
    --avx2

!ls -lh {ONNX_DIR}/int8/

In [ ]:
# @title 14. Generate vocab.txt and label_map.json
#
# These are the two companion files the app downloads alongside the .onnx.
# vocab.txt  — one token per line, index = token ID (matches input_ids)
# label_map.json — {"0": "O", "1": "B-MER", ...}

RELEASE_DIR.mkdir(exist_ok=True)

# vocab.txt — copy from tokenizer (DistilBERT uses WordPiece vocab)
src_vocab = OUTPUT_DIR / "vocab.txt"
dst_vocab = RELEASE_DIR / "vocab.txt"
shutil.copy(src_vocab, dst_vocab)
print(f"vocab.txt: {sum(1 for _ in open(dst_vocab))} tokens")

# label_map.json
label_map = {str(i): label for i, label in ID2LABEL.items()}
with open(RELEASE_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)
print("label_map.json:", label_map)

# Copy quantized model
shutil.copy(
    ONNX_DIR / "int8" / "model_quantized.onnx",
    RELEASE_DIR / "receipt-classifier.onnx",
)

!ls -lh {RELEASE_DIR}

In [ ]:
# @title 15. Validate output shapes match ReceiptONNXService.ts expectations
#
# ReceiptONNXService expects:
#   inputs:  input_ids [1, seq_len] int64
#            attention_mask [1, seq_len] int64
#   outputs: logits [1, seq_len, 10] float32

import onnxruntime as ort

sess = ort.InferenceSession(str(RELEASE_DIR / "receipt-classifier.onnx"))

print("Inputs:")
for inp in sess.get_inputs():
    print(f"  {inp.name}: {inp.type} {inp.shape}")

print("Outputs:")
for out in sess.get_outputs():
    print(f"  {out.name}: {out.type} {out.shape}")

# Run a quick inference sanity check
sample_text = "STARBUCKS COFFEE 12/05/2025 Total $15.60 Tax $0.87"
enc = tokenizer(sample_text.split(), is_split_into_words=True, return_tensors="np")

outputs = sess.run(
    None,
    {
        "input_ids": enc["input_ids"].astype(np.int64),
        "attention_mask": enc["attention_mask"].astype(np.int64),
    },
)

logits = outputs[0]  # [1, seq_len, 10]
preds = np.argmax(logits[0], axis=-1)
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

print("\nSample inference:")
for tok, pred_id in zip(tokens, preds):
    print(f"  {tok:<20} {ID2LABEL[pred_id]}")

assert logits.shape[2] == NUM_LABELS, f"Expected {NUM_LABELS} classes, got {logits.shape[2]}"
print(f"\n✅  Output shape {logits.shape} matches ReceiptONNXService.ts expectations.")

## 16. Package for GitHub Release

Download the 3 files from `release/` and attach them to a GitHub Release in `atlas-ledger-models`.

### Option A — GitHub CLI (if running locally)

```bash
cd release/
gh release create receipt-ner-v1.0 \
  receipt-classifier.onnx \
  vocab.txt \
  label_map.json \
  --repo buzybee83/atlas-ledger-models \
  --title "Receipt NER v1.0" \
  --notes "DistilBERT INT8 ONNX — BIO token classifier for receipt field extraction."
```

### Option B — Colab download + manual upload

```python
from google.colab import files
files.download('release/receipt-classifier.onnx')
files.download('release/vocab.txt')
files.download('release/label_map.json')
```
Then upload manually at: `github.com/buzybee83/atlas-ledger-models/releases/new`

### After upload — set the URL in the app

In `src/services/ReceiptONNXService.ts`, set:

```typescript
const RECEIPT_MODEL_DOWNLOAD_URL =
  'https://github.com/buzybee83/atlas-ledger-models/releases/download/receipt-ner-v1.0/receipt-classifier.onnx';
```

The app downloads this on first use and caches it at `{documentDirectory}/models/receipt-classifier.onnx`.

In [ ]:
# @title 16b. (Colab) Download release files
from google.colab import files
files.download('release/receipt-classifier.onnx')
files.download('release/vocab.txt')
files.download('release/label_map.json')